**Writing Files in Disk :**
- What are the modes available in dataframe writer ?
- What is PartitionBy and bucketBy ?
- How to write data into multiple partitions ?

In [0]:
# DataFrameWriter.format().option().mode().partitionBy().bucketBy().save()
# Mode :
#   append - 
#   Overwrite - 
#   errorIfExists - default
#   ignore - 



In [0]:
emp_df = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema","true")\
    .load("/Volumes/workspace/mde/testdata/employee_write.csv")

In [0]:
emp_df.write.format("csv")\
    .option("header",True)\
    .mode("overwrite")\
    .save("/Volumes/workspace/mde/testdata/csv_write")

# for CSV -> if you skip the header option, Spark writes only the data rows and does not include the column names in the output file.
# pass mode separately not inside option.

Repartition :

In [0]:
emp_df.repartition(3).write.format("csv")\
    .option("header",True)\
    .mode("overwrite")\
    .save("/Volumes/workspace/mde/testdata/csv_write_repartition/")

# for CSV -> if you skip the header option, Spark writes only the data rows and does not include the column names in the output file.
# pass mode separately not inside option.

**Partitioning & Bucketing :**
- What is partitioning in spark ?
- What is bucketing in spark ?
- Why do we need these two ?
- when to use partitioning and Bucketing ?

In [0]:
# Partitioning : Organizing data into folders based on column values(low cardinal) so Spark can skip irrelevant data while reading.
#     -> used for Partition Pruning(filtering) -> huge performance improvement.

# Bucketing : Organizing data into a fixed number of hash-based buckets to reduce shuffle and improve join performance
#     -> USE When : Large joins happen frequently.
#                   High-cardinality columns exist (many unique values).
#                   Partitioning would create too many folders.

PartitionBy :

In [0]:
partitioned_emp_df = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema","true")\
    .load("/Volumes/workspace/mde/testdata/employee_write.csv")

In [0]:
partitioned_emp_df.write.format("csv")\
    .option("header",True)\
    .mode("overwrite")\
    .partitionBy("address")\
    .save("/Volumes/workspace/mde/testdata/partitioned_by_address/")
# can pass multiple columns in argument in proper order.

BucketBy :

In [0]:
partitioned_emp_df.write.format("Delta")\
    .option("header",True)\
    .mode("overwrite")\
    .bucketBy(3,"id")\
    .saveAsTable("bucket_by_id_table")
# Bucketing requires Spark to store metadata such as -> Number of buckets ,Bucket column(s)
# So requires saveAsTable() because Spark needs table metadata to track bucket information and use it for optimizations such as joins.

In [0]:
%fs
ls /Volumes/workspace/mde/testdata/partitioned_by_address/